# Week 6: Unsupervised Learning & Dimensionality Reduction
## Notebook 06: DBSCAN (Density-Based Spatial Clustering) from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Standard K-Means clustering has severe limitations:
1. It assumes clusters are spherical and convex (fails completely on concentric circles, moons, or arbitrary elongated paths).
2. It requires you to know $K$ in advance.
3. It forces every outlier into a cluster.

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**:
- Discovers clusters of **arbitrary non-linear shapes**.
- Does **NOT require specifying the number of clusters $K$**.
- Automatically identifies and isolates **Noise / Outliers** (labeled as $-1$).

---

### 2. Mathematical Foundations: Density Reachability
Two hyperparameters govern DBSCAN:
- $\epsilon$ (Epsilon): Radius of neighborhood around point $p$:
  $$N_\epsilon(p) = \{ q \in D \mid \text{dist}(p, q) \le \epsilon \}$$
- $\text{MinPts}$: Minimum points required inside $N_\epsilon(p)$ to form a dense region.

#### Point Classifications:
1. **Core Point:** $|N_\epsilon(p)| \ge \text{MinPts}$.
2. **Border Point:** $|N_\epsilon(p)| < \text{MinPts}$, but lies inside $N_\epsilon(c)$ of a Core Point $c$.
3. **Noise Point (Outlier):** Neither Core nor Border.

In [ ]:
import numpy as np

# Step 1: Create Synthetic Non-Convex Dataset (Concentric Circles + Outliers)
np.random.seed(42)
n_samples = 80

# Inner ring
theta1 = np.linspace(0, 2 * np.pi, n_samples)
r1 = 1.0 + np.random.randn(n_samples) * 0.05
ring_inner = np.column_stack([r1 * np.cos(theta1), r1 * np.sin(theta1)])

# Outer ring
theta2 = np.linspace(0, 2 * np.pi, n_samples)
r2 = 3.0 + np.random.randn(n_samples) * 0.05
ring_outer = np.column_stack([r2 * np.cos(theta2), r2 * np.sin(theta2)])

# Outlier points
noise = np.array([[0.0, 0.0], [4.5, 4.5], [-4.0, 3.5]])

X = np.vstack([ring_inner, ring_outer, noise])
print(f"Total dataset points: {len(X)} (Inner ring, Outer ring, and isolated outliers)")

# Step 2: Implement DBSCAN from Scratch
class DBSCANScratch:
    def __init__(self, eps=0.6, min_pts=4):
        self.eps = eps
        self.min_pts = min_pts

    def fit(self, X):
        N = len(X)
        labels = np.full(N, -1)  # -1 represents Noise
        cluster_id = 0
        visited = np.zeros(N, dtype=bool)

        # Precompute Euclidean distance matrix
        dist_matrix = np.linalg.norm(X[:, np.newaxis, :] - X[np.newaxis, :, :], axis=2)

        for i in range(N):
            if visited[i]:
                continue
            visited[i] = True
            
            # Find neighbors within eps
            neighbors = np.where(dist_matrix[i] <= self.eps)[0]
            
            if len(neighbors) < self.min_pts:
                labels[i] = -1  # Mark as Noise for now
            else:
                # Expand cluster from Core point
                labels[i] = cluster_id
                queue = list(neighbors[neighbors != i])
                
                while queue:
                    neighbor_idx = queue.pop(0)
                    if not visited[neighbor_idx]:
                        visited[neighbor_idx] = True
                        sub_neighbors = np.where(dist_matrix[neighbor_idx] <= self.eps)[0]
                        if len(sub_neighbors) >= self.min_pts:
                            queue.extend(sub_neighbors[~np.isin(sub_neighbors, queue)])
                            
                    if labels[neighbor_idx] == -1:
                        labels[neighbor_idx] = cluster_id
                        
                cluster_id += 1
                
        return labels

dbscan = DBSCANScratch(eps=0.7, min_pts=4)
labels = dbscan.fit(X)

unique_clusters = set(labels)
print(f"\nDBSCAN Discovered Clusters: {unique_clusters} (-1 is Noise)")
print(f"Cluster 0 size (Inner Ring): {np.sum(labels == 0)}")
print(f"Cluster 1 size (Outer Ring): {np.sum(labels == 1)}")
print(f"Identified Outlier Points (Noise): {np.sum(labels == -1)}")
print("Outlier Indices:", np.where(labels == -1)[0])

### 3. Step-by-Step Code Walkthrough
1. **Distance Neighborhoods:** Points within radius $\epsilon=0.7$ form graph edges.
2. **Cluster Expansion via BFS:** When a core point is found, its density-reachable neighbors are queued and absorbed into the current cluster.
3. **Flawless Outlier Handling:** The 3 isolated points failed the density threshold and were cleanly flagged as noise ($-1$) without distorting the ring clusters.

---

### 4. Industry Applications & Trade-offs
- **GPS Telematics & Anomaly Detection:** Clustering vehicle stop locations and detecting fraudulent delivery deviations.
- **Trade-off:** DBSCAN struggles when clusters have varying densities (e.g. one very dense cluster and one sparse cluster). In such cases, **HDBSCAN** (Hierarchical DBSCAN) is preferred.

---

### 5. Top Interview Questions & Answers
**Q1: How do you choose the $\epsilon$ parameter for DBSCAN?**
> **Answer:** Use the **$k$-distance graph**: compute the distance from every point to its $k$-th nearest neighbor ($k = \text{MinPts}$), sort the distances in ascending order, and plot them. The optimal $\epsilon$ corresponds to the "elbow" or "knee" point where distances sharply increase.

**Q2: What is the computational complexity of DBSCAN?**
> **Answer:** With a brute-force distance matrix, complexity is $O(N^2)$. Using spatial index trees like $k$-d trees or Ball trees for neighborhood queries reduces complexity to $O(N \log N)$, making it scalable to large low-dimensional datasets.